In [ ]:
import os, re, json, time, shutil, hashlib, zipfile, platform
from pathlib import Path

STEP2_DIR = Path("outputs/step2_index")
STEP3_DIR = Path("outputs/step3_dataset")
STEP4_DIR = Path("outputs/step4_qlora")
STEP5_CFG = Path("outputs/step5_pipeline/pipeline_config.json")
STEP6_DIR = Path("outputs/step6_final")
BUNDLE_DIR = Path("outputs/b1_bundle")
ZIP_PATH = Path("outputs/b1_bundle.zip")
DRIVE_DIR = None            # ex. Path("/content/drive/MyDrive/b1") après drive.mount : le zip y est aussi copié (la session Colab est éphémère)
DEFAULT_CONFIG = "C"        # configuration proposée par défaut dans l'interface : "C" (avec adaptateur) ou "B" — à décider d'après les résultats de l'étape 6

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""): h.update(block)
    return h.hexdigest()

## 1. Contrôles : le bundle correspond-il à ce qui a été évalué ?

In [ ]:
assert DEFAULT_CONFIG in ("B", "C")
ADAPTER_SRC = STEP4_DIR / "adapter"
need = [STEP2_DIR / "index_config.json", STEP2_DIR / "chunks.jsonl", STEP2_DIR / "faiss.index", STEP3_DIR / "rag_prompt.json",
        STEP3_DIR / "dataset_config.json", STEP5_CFG, ADAPTER_SRC / "adapter_model.safetensors", ADAPTER_SRC / "adapter_config.json"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"fichiers manquants : {missing}"

pcfg = json.load(open(STEP5_CFG, encoding="utf-8"))
rag_prompt = json.load(open(STEP3_DIR / "rag_prompt.json", encoding="utf-8"))
index_cfg = json.load(open(STEP2_DIR / "index_config.json", encoding="utf-8"))
adapter_cfg = json.load(open(ADAPTER_SRC / "adapter_config.json", encoding="utf-8"))

adapter_sha = sha256_file(ADAPTER_SRC / "adapter_model.safetensors")
prompt_sha = hashlib.sha256(json.dumps(rag_prompt, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
checks = [("adaptateur = celui de l'étape 5", adapter_sha == pcfg["adapter_sha256"]),
          ("prompt = celui de l'étape 5", prompt_sha == pcfg["prompt_sha256"]),
          ("k / embedder = étape 5", (index_cfg["k_final"], index_cfg["embedder"]) == (pcfg["retrieval"]["k"], pcfg["retrieval"]["embedder"]))]
m6 = STEP6_DIR / "meta_step6.json"
if m6.exists():
    meta6 = json.load(open(m6, encoding="utf-8"))
    checks += [("adaptateur = celui évalué à l'étape 6", meta6["adapter_sha256"] == adapter_sha), ("prompt = celui évalué à l'étape 6", meta6["prompt_sha256"] == prompt_sha)]
else:
    print("⚠️ étape 6 non trouvée : on ne peut pas prouver que le bundle est celui qui a été évalué.")
for name, ok in checks: print("OK    " if ok else "ÉCHEC ", name)
assert all(ok for _, ok in checks), "STOP — les artefacts ne correspondent pas à ce qui a été validé/évalué"
print("adaptateur LoRA : r =", adapter_cfg.get("r"), "| alpha =", adapter_cfg.get("lora_alpha"), "| modules =", adapter_cfg.get("target_modules"),
      "| base déclarée =", adapter_cfg.get("base_model_name_or_path"))

## 2. Construction du bundle + manifeste (SHA-256 de chaque fichier)
`bundle_manifest.json` est lu par l'interface au démarrage. L'`embeddings.npy` n'est pas copié (l'index FAISS suffit).

In [ ]:
if BUNDLE_DIR.exists(): shutil.rmtree(BUNDLE_DIR)
copies = [(STEP2_DIR / "index_config.json", "step2_index"), (STEP2_DIR / "chunks.jsonl", "step2_index"), (STEP2_DIR / "faiss.index", "step2_index"),
          (STEP3_DIR / "rag_prompt.json", "step3_dataset"), (STEP3_DIR / "dataset_config.json", "step3_dataset"), (STEP5_CFG, "step5_pipeline")]
if (STEP3_DIR / "val.jsonl").exists(): copies.append((STEP3_DIR / "val.jsonl", "step3_dataset"))      # contrôle « prompt de production = prompt d'entraînement » au démarrage
for name in ("metrics_final.csv", "meta_step6.json", "paired_tests.csv"):
    if (STEP6_DIR / name).exists(): copies.append((STEP6_DIR / name, "step6_final"))
for src, sub in copies:
    (BUNDLE_DIR / sub).mkdir(parents=True, exist_ok=True); shutil.copy2(src, BUNDLE_DIR / sub / src.name)
(BUNDLE_DIR / "adapter").mkdir(parents=True, exist_ok=True)
for f in ADAPTER_SRC.iterdir():
    skip = f.name.startswith(("optimizer", "scheduler", "rng_state", "trainer_state")) or f.suffix in (".pt", ".pth")      # états d'entraînement : inutiles en production
    if f.is_file() and not skip:
        shutil.copy2(f, BUNDLE_DIR / "adapter" / f.name)

try:
    import torch, transformers, peft
    versions = dict(torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__)
except Exception:
    versions = {}
manifest = {
    "name": "B1 — Sentinelle (assistant réglementaire sourcé)", "created": time.strftime("%Y-%m-%d %H:%M:%S"), "default_config": DEFAULT_CONFIG,
    "base_model": pcfg["model"], "quantization": pcfg["quantization"], "base_model_note": "non inclus : téléchargé depuis Hugging Face au premier lancement (≈ 15 Go)",
    "adapter": {"sha256": adapter_sha, "r": adapter_cfg.get("r"), "alpha": adapter_cfg.get("lora_alpha"), "target_modules": adapter_cfg.get("target_modules"),
                "declared_base": adapter_cfg.get("base_model_name_or_path")},
    "retrieval": pcfg["retrieval"], "similarity_floor": pcfg["similarity_floor"], "prompt_sha256": prompt_sha, "decoding": pcfg["decoding"], "enforce_citation": pcfg["enforce_citation"],
    "versions_at_export": versions, "python": platform.python_version(), "evaluated_in_step6": m6.exists(),
    "files": {str(p.relative_to(BUNDLE_DIR)): sha256_file(p) for p in sorted(BUNDLE_DIR.rglob("*")) if p.is_file()},
}
json.dump(manifest, open(BUNDLE_DIR / "bundle_manifest.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
tot = sum(p.stat().st_size for p in BUNDLE_DIR.rglob("*") if p.is_file()) / 1e6
print(f"bundle : {len(manifest['files'])} fichiers, {tot:.1f} Mo → {BUNDLE_DIR.resolve()}")
for k, v in manifest["files"].items(): print(f"  {v[:10]}…  {k}")

## 3. Vérification d'intégrité du bundle, archive et copie hors de la session
La vérification relit chaque fichier et compare son SHA-256 au manifeste (à refaire après tout transfert Drive / zip).

In [ ]:
def verify_bundle(bdir):
    man = json.load(open(Path(bdir) / "bundle_manifest.json", encoding="utf-8")); bad = []
    for rel, sha in man["files"].items():
        p = Path(bdir) / rel
        if not p.exists() or sha256_file(p) != sha: bad.append(rel)
    return bad

assert not verify_bundle(BUNDLE_DIR), verify_bundle(BUNDLE_DIR)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(BUNDLE_DIR.rglob("*")):
        if p.is_file(): z.write(p, Path("b1_bundle") / p.relative_to(BUNDLE_DIR))
print(f"archive : {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} Mo) — intégrité du dossier vérifiée")
if DRIVE_DIR:
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True); shutil.copy2(ZIP_PATH, Path(DRIVE_DIR) / ZIP_PATH.name); print("copie sur Drive :", Path(DRIVE_DIR) / ZIP_PATH.name)
try:
    from google.colab import files; files.download(str(ZIP_PATH))      # téléchargement navigateur (Colab)
except Exception:
    print("(hors Colab : récupérer l'archive à l'emplacement ci-dessus)")